# Trening sieci `VegetationLSTM` (NDVI & NDMI) w Google Colab na GPU

Ten notatnik ładuje kod oraz lekkie tabele `.parquet`, trenuje model LSTM (domyślnie w trybie **`pixel`** 10 m z danymi glebowymi H3 albo w trybie **`field`**) na GPU w Colabie i zapisuje wytrenowane wagi oraz raporty.

## 1. Włączenie GPU w Colabie
1. Otwórz menu **Środowisko wykonawcze** -> **Zmień typ środowiska wykonawczego**.
2. Wybierz **T4 GPU** (lub lepszy akcelerator GPU) i kliknij **Zapisz**.

## 2. Dwa sposoby podpięcia plików (`ml/` + `data/ml/*.parquet`, ~6 MB łącznie)
- **Opcja A — Google Drive (`USE_GOOGLE_DRIVE = True`, domyślna):**
  Wrzuć na `MyDrive/field-under-pressure-satops/` foldery `ml/` oraz `data/ml/*.parquet`. Jeśli `drive.mount()` zgłosi błąd **`400 / Error: credential propagation was unsuccessful`**, to:
  1. uruchom notatnik w **przeglądarce Google Chrome** na **jednym koncie Google** z włączonymi plikami cookie stron trzecich,
  2. albo kliknij ikonę folderu po lewej -> **Zamontuj Dysk** zanim odpalisz komórkę,
  3. albo skorzystaj z Opcji B poniżej (szczególnie w rozszerzeniu **Colab w VS Code**, gdzie logowanie OAuth do Drive często nie przechodzi).
- **Opcja B — bez Google Drive przez plik ZIP (`USE_GOOGLE_DRIVE = False`):**
  Ustaw w komórce 2 `USE_GOOGLE_DRIVE = False`, spakuj lokalnie `zip -r satops_colab.zip ml data/ml -x 'ml/__pycache__/*' 'ml/checkpoints/*'` i przeciągnij `satops_colab.zip` do panelu **Pliki (`/content/`)** po lewej. Wyniki spakują się do `/content/satops_work/wyniki_<mode>_<timestamp>.zip`.

## 3. Gdzie trafią wyniki
- Zawsze powstaje lokalna paczka ZIP: `/content/satops_work/wyniki_<mode>_<timestamp>.zip` (checkpoint `.pt`, `training_history.csv`, `training_summary.json`, `learning_curves.png`, `val_horizon_metrics.csv`, `sample_val_predictions.csv`).
- W trybie `USE_GOOGLE_DRIVE = True` wyniki trafiają dodatkowo na **`MyDrive/field-under-pressure-satops/wyniki/run_<mode>_<timestamp>/`** oraz do **`MyDrive/field-under-pressure-satops/ml/checkpoints/`**.

In [ ]:
from datetime import datetime
import os
from pathlib import Path
import shutil
import sys
import zipfile

# === Wybór źródła danych: True = Google Drive, False = paczka satops_colab.zip w /content/ ===
USE_GOOGLE_DRIVE = False
LOCAL_ZIP_PATH = Path("/content/satops_colab.zip")

# === Ścieżki na Google Drive oraz szybkim dysku lokalnym Colab SSD ===
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/field-under-pressure-satops")
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data" / "ml"
DRIVE_RESULTS_ROOT = DRIVE_PROJECT_DIR / "wyniki"

LOCAL_WORK_DIR = Path("/content/satops_work")
LOCAL_DATA_DIR = LOCAL_WORK_DIR / "data" / "ml"
LOCAL_OUT_DIR = LOCAL_WORK_DIR / "outputs"

IN_COLAB = "google.colab" in sys.modules
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE:
    if IN_COLAB and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive

        try:
            drive.mount("/content/drive", force_remount=False)
        except Exception as exc:
            raise RuntimeError(
                f"Nie udało się zamontować Google Drive: {exc}\n\n"
                "Możesz to obejść na 3 sposoby:\n"
                "1) Otwórz notatnik w przeglądarce Chrome (jedno konto Google, włączone cookies, bez incognito).\n"
                "2) Zamontuj Dysk ręcznie ikoną w panelu Pliki po lewej stronie Colaba.\n"
                "3) Zmień `USE_GOOGLE_DRIVE = False` powyżej (działa też w VS Code Colab), spakuj na Macu:\n"
                "   zip -r satops_colab.zip ml data/ml -x 'ml/__pycache__/*' 'ml/checkpoints/*'\n"
                "   i przeciągnij plik `satops_colab.zip` do /content/ w panelu Pliki."
            ) from exc

    if not DRIVE_PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"Nie znaleziono folderu projektu na Drive: {DRIVE_PROJECT_DIR}\n"
            "Wgraj foldery `ml/` oraz `data/ml/` pod wskazaną ścieżkę na Google Drive "
            "albo przełącz `USE_GOOGLE_DRIVE = False` i użyj paczki satops_colab.zip."
        )
    if not (DRIVE_PROJECT_DIR / "ml").exists():
        raise FileNotFoundError(f"Brak folderu z kodem: {DRIVE_PROJECT_DIR / 'ml'}")
    if not DRIVE_DATA_DIR.exists():
        raise FileNotFoundError(f"Brak folderu z tabelami Parquet: {DRIVE_DATA_DIR}")

    shutil.copytree(
        DRIVE_PROJECT_DIR / "ml",
        LOCAL_WORK_DIR / "ml",
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "checkpoints"),
    )
    parquet_files = sorted(DRIVE_DATA_DIR.glob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError(f"Brak plików *.parquet w {DRIVE_DATA_DIR}")
    for src_pq in parquet_files:
        shutil.copy2(src_pq, LOCAL_DATA_DIR / src_pq.name)
else:
    if not LOCAL_ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Nie znaleziono pliku ZIP: {LOCAL_ZIP_PATH}\n"
            "Utwórz paczkę w katalogu projektu na Macu:\n"
            "  zip -r satops_colab.zip ml data/ml -x 'ml/__pycache__/*' 'ml/checkpoints/*'\n"
            f"i prześlij ją do {LOCAL_ZIP_PATH} (panel Pliki po lewej w Colabie)."
        )
    with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zf:
        zf.extractall(LOCAL_WORK_DIR)

    if not (LOCAL_WORK_DIR / "ml").exists():
        nested_dirs = [
            d for d in LOCAL_WORK_DIR.iterdir()
            if d.is_dir() and d.name != "__MACOSX" and (d / "ml").exists()
        ]
        if nested_dirs:
            nested = nested_dirs[0]
            shutil.copytree(nested / "ml", LOCAL_WORK_DIR / "ml", dirs_exist_ok=True)
            if (nested / "data" / "ml").exists():
                for pq in (nested / "data" / "ml").glob("*.parquet"):
                    shutil.copy2(pq, LOCAL_DATA_DIR / pq.name)

    if not (LOCAL_WORK_DIR / "ml").exists():
        raise FileNotFoundError(f"W archiwum {LOCAL_ZIP_PATH} brakuje folderu `ml/`.")
    if not sorted(LOCAL_DATA_DIR.glob("*.parquet")):
        raise FileNotFoundError(f"W archiwum {LOCAL_ZIP_PATH} brakuje plików `data/ml/*.parquet`.")

imported_parquets = sorted(LOCAL_DATA_DIR.glob("*.parquet"))
print(f"Moduł ml/ gotowy -> {LOCAL_WORK_DIR / 'ml'}")
print(f"Dostępne pliki Parquet ({len(imported_parquets)}) w {LOCAL_DATA_DIR}:")
for dst_pq in imported_parquets:
    size_mb = dst_pq.stat().st_size / (1024 * 1024)
    print(f"  - {dst_pq.name:35s} ({size_mb:.2f} MB)")

work_root_str = str(LOCAL_WORK_DIR)
if work_root_str not in sys.path:
    sys.path.insert(0, work_root_str)
os.chdir(LOCAL_WORK_DIR)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Katalog roboczy: {Path.cwd()} | Znacznik uruchomienia: {RUN_TIMESTAMP}")

RuntimeError: Nie udało się zamontować Google Drive: User cancelled dfs_ephemeral authorization

Możesz to obejść na 3 sposoby:
1) Otwórz notatnik w przeglądarce Chrome (jedno konto Google, włączone cookies, bez incognito).
2) Zamontuj Dysk ręcznie ikoną w panelu Pliki po lewej stronie Colaba.
3) Zmień `USE_GOOGLE_DRIVE = False` powyżej (działa też w VS Code Colab), spakuj na Macu:
   zip -r satops_colab.zip ml data/ml -x 'ml/__pycache__/*' 'ml/checkpoints/*'
   i przeciągnij plik `satops_colab.zip` do /content/ w panelu Pliki.

In [ ]:
import importlib
import subprocess
import sys

CORE_PACKAGES = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("pyarrow", "pyarrow"),
    ("matplotlib", "matplotlib"),
]
missing_pkgs = [pip_name for mod_name, pip_name in CORE_PACKAGES if importlib.util.find_spec(mod_name) is None]
if missing_pkgs:
    print(f"Instaluję brakujące pakiety bazowe: {missing_pkgs}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])

# Ustaw na True tylko wtedy, gdy chcesz dodatkowo uruchamiać predykcję na rastrach GeoTIFF w Colabie
INSTALL_GEO_DEPS = False
if INSTALL_GEO_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio", "geopandas", "duckdb"])

import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_props = torch.cuda.get_device_properties(0)
    vram_gb = gpu_props.total_memory / (1024 ** 3)
    print(f"[OK] Akcelerator GPU aktywny: {gpu_props.name} ({vram_gb:.1f} GB VRAM) | PyTorch {torch.__version__}")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print(f"[OK] Akcelerator Apple MPS aktywny | PyTorch {torch.__version__}")
else:
    DEVICE = torch.device("cpu")
    print(
        f"[UWAGA] Brak GPU (uruchomiono na CPU) | PyTorch {torch.__version__}.\n"
        "W Colabie włącz: Środowisko wykonawcze -> Zmień typ środowiska wykonawczego -> T4 GPU."
    )

In [ ]:
import pandas as pd

# === Parametry treningu ===
TRAIN_MODE = "pixel"          # "pixel" (10m + H3) lub "field" (średnie po polu)
EPOCHS = 30
BATCH_SIZE = 512
ANCHOR_STRIDE = 2             # Przesuw okna w dniach (w pixel zalecane 2-4, w field 1)
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.15
VAL_RATIO = 0.2
SPLIT_MODE = "field"          # "field" (zero przecieku między polami) lub "time"
TARGET_MODE = "observed"      # "observed" (prawdziwe przeloty S2) lub "interpolated"
MAX_INTERP_GAP_DAYS = 10
MIN_HIST_OBS = 2
SEED = 42
COPY_TO_CHECKPOINTS = True    # Skopiuj najlepszy .pt także do DRIVE_PROJECT_DIR/ml/checkpoints/

if TRAIN_MODE == "pixel":
    MASTER_FILE = LOCAL_DATA_DIR / "pixel_master.parquet"
    SAT_FILE = LOCAL_DATA_DIR / "pixel_satellite_indices.parquet"
    ENTITY_COL = "pixel_id"
    CKPT_FILENAME = "vegetation_lstm_pixel.pt"
elif TRAIN_MODE == "field":
    MASTER_FILE = LOCAL_DATA_DIR / "fields_master.parquet"
    SAT_FILE = LOCAL_DATA_DIR / "satellite_indices.parquet"
    ENTITY_COL = "field_id"
    CKPT_FILENAME = "vegetation_lstm.pt"
else:
    raise ValueError(f"Nieznany TRAIN_MODE={TRAIN_MODE!r}; wybierz 'pixel' albo 'field'.")

WEATHER_FILE = LOCAL_DATA_DIR / "weather_daily.parquet"
for req_path in (MASTER_FILE, SAT_FILE, WEATHER_FILE):
    if not req_path.exists():
        raise FileNotFoundError(f"Brak wymaganego pliku danych: {req_path}")

_m_df = pd.read_parquet(MASTER_FILE, columns=["field_id"])
_s_df = pd.read_parquet(SAT_FILE, columns=["date"])
_w_df = pd.read_parquet(WEATHER_FILE, columns=["date"])

print(f"Tryb treningu: {TRAIN_MODE} (entity_col={ENTITY_COL}) -> checkpoint: {CKPT_FILENAME}")
print(f"  - {MASTER_FILE.name}: {len(_m_df)} encji ({_m_df['field_id'].astype(str).nunique()} pól)")
print(f"  - {SAT_FILE.name}: {len(_s_df)} obserwacji ({_s_df['date'].min()} .. {_s_df['date'].max()})")
print(f"  - {WEATHER_FILE.name}: {len(_w_df)} wierszy ({_w_df['date'].min()} .. {_w_df['date'].max()})")
del _m_df, _s_df, _w_df

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

from ml.config import (
    FUTURE_FEATURE_NAMES,
    HIST_FEATURE_NAMES,
    STATIC_FEATURE_NAMES,
    TARGET_COLS,
    ModelConfig,
    TrainConfig,
)
from ml.dataset import FeatureScaler, FieldTimeSeriesDataset, split_indices
from ml.model import VegetationLSTM, compute_masked_metrics, masked_smooth_l1_loss

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_cfg = ModelConfig(
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
)
train_cfg = TrainConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    val_ratio=VAL_RATIO,
    split_mode=SPLIT_MODE,
    target_mode=TARGET_MODE,
    max_interp_gap_days=MAX_INTERP_GAP_DAYS,
    min_hist_obs=MIN_HIST_OBS,
    seed=SEED,
)

t_ds0 = time.perf_counter()
dataset = FieldTimeSeriesDataset(
    fields_master_parquet=MASTER_FILE,
    satellite_indices_parquet=SAT_FILE,
    weather_daily_parquet=WEATHER_FILE,
    history_days=model_cfg.history_days,
    forecast_days=model_cfg.forecast_days,
    target_mode=train_cfg.target_mode,
    max_interp_gap_days=train_cfg.max_interp_gap_days,
    min_hist_obs=train_cfg.min_hist_obs,
    entity_col=ENTITY_COL,
    anchor_stride=ANCHOR_STRIDE,
)
train_idx, val_idx = split_indices(
    dataset,
    val_ratio=train_cfg.val_ratio,
    split_mode=train_cfg.split_mode,
    seed=train_cfg.seed,
)

if SPLIT_MODE == "field":
    train_fields = {dataset.samples[i]["field_id"] for i in train_idx}
    val_fields = {dataset.samples[i]["field_id"] for i in val_idx}
    field_overlap = len(train_fields & val_fields)
    assert field_overlap == 0, f"Wykryto przeciek pól między train i val: {field_overlap}"
    print(
        f"[DATASET] Zbudowano w {time.perf_counter() - t_ds0:.1f}s | "
        f"okna train={len(train_idx)} ({len(train_fields)} pól), "
        f"val={len(val_idx)} ({len(val_fields)} pól), przeciek pól=0"
    )
else:
    print(
        f"[DATASET] Zbudowano w {time.perf_counter() - t_ds0:.1f}s | "
        f"okna train={len(train_idx)}, val={len(val_idx)} (split_mode=time)"
    )

scaler = FeatureScaler().fit_from_dataset(dataset, train_idx)
dataset.scaler = scaler

use_cuda = DEVICE.type == "cuda"
num_workers = 2 if use_cuda else 0
train_loader = DataLoader(
    Subset(dataset, train_idx),
    batch_size=train_cfg.batch_size,
    shuffle=True,
    pin_memory=use_cuda,
    num_workers=num_workers,
)
val_loader = DataLoader(
    Subset(dataset, val_idx),
    batch_size=train_cfg.batch_size,
    shuffle=False,
    pin_memory=use_cuda,
    num_workers=num_workers,
)

model = VegetationLSTM(model_cfg).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

CKPT_OUT_PATH = LOCAL_OUT_DIR / CKPT_FILENAME
best_val_loss = float("inf")
best_metrics = {}
best_epoch = 0
history_rows = []

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[TRAIN] Start na {DEVICE} | parametry={n_params:,} | epoki={train_cfg.epochs}, batch={train_cfg.batch_size}")

t_train_start = time.perf_counter()
for epoch in range(1, train_cfg.epochs + 1):
    t_ep0 = time.perf_counter()
    model.train()
    train_loss_sum = 0.0
    train_valid = 0

    for batch in train_loader:
        x_hist = batch["x_hist"].to(DEVICE, non_blocking=use_cuda)
        x_future = batch["x_future"].to(DEVICE, non_blocking=use_cuda)
        x_static = batch["x_static"].to(DEVICE, non_blocking=use_cuda)
        last_targets = batch["last_targets"].to(DEVICE, non_blocking=use_cuda)
        y_target = batch["y_target"].to(DEVICE, non_blocking=use_cuda)
        y_mask = batch["y_mask"].to(DEVICE, non_blocking=use_cuda)

        optimizer.zero_grad(set_to_none=True)
        pred = model(x_hist, x_future, x_static, last_targets)
        loss = masked_smooth_l1_loss(pred, y_target, y_mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        optimizer.step()

        n_valid = int((y_mask > 0).sum().item())
        if n_valid > 0:
            train_loss_sum += float(loss.item()) * n_valid
            train_valid += n_valid

    train_loss = train_loss_sum / max(1, train_valid)

    model.eval()
    val_loss_sum = 0.0
    val_valid = 0
    val_preds, val_targets, val_masks = [], [], []
    with torch.inference_mode():
        for batch in val_loader:
            x_hist = batch["x_hist"].to(DEVICE, non_blocking=use_cuda)
            x_future = batch["x_future"].to(DEVICE, non_blocking=use_cuda)
            x_static = batch["x_static"].to(DEVICE, non_blocking=use_cuda)
            last_targets = batch["last_targets"].to(DEVICE, non_blocking=use_cuda)
            y_target = batch["y_target"].to(DEVICE, non_blocking=use_cuda)
            y_mask = batch["y_mask"].to(DEVICE, non_blocking=use_cuda)

            pred = model(x_hist, x_future, x_static, last_targets)
            loss = masked_smooth_l1_loss(pred, y_target, y_mask)
            n_valid = int((y_mask > 0).sum().item())
            if n_valid > 0:
                val_loss_sum += float(loss.item()) * n_valid
                val_valid += n_valid
            val_preds.append(pred.cpu())
            val_targets.append(y_target.cpu())
            val_masks.append(y_mask.cpu())

    val_loss = val_loss_sum / max(1, val_valid)
    val_metrics = compute_masked_metrics(
        torch.cat(val_preds, dim=0),
        torch.cat(val_targets, dim=0),
        torch.cat(val_masks, dim=0),
    )
    cur_lr = float(optimizer.param_groups[0]["lr"])
    scheduler.step(val_loss)
    epoch_sec = time.perf_counter() - t_ep0

    history_rows.append(
        {
            "epoch": epoch,
            "lr": cur_lr,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "ndvi_mae": val_metrics["ndvi_mae"],
            "ndmi_mae": val_metrics["ndmi_mae"],
            "ndvi_rmse": val_metrics["ndvi_rmse"],
            "ndmi_rmse": val_metrics["ndmi_rmse"],
            "epoch_sec": epoch_sec,
        }
    )

    best_tag = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_metrics = dict(val_metrics)
        best_epoch = epoch
        best_tag = " [BEST]"
        torch.save(
            {
                "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "model_config": model_cfg.to_dict(),
                "train_config": train_cfg.to_dict(),
                "scaler": scaler.to_dict(),
                "val_metrics": best_metrics,
                "entity_mode": TRAIN_MODE,
                "feature_schema": {
                    "hist": list(HIST_FEATURE_NAMES),
                    "future": list(FUTURE_FEATURE_NAMES),
                    "static": list(STATIC_FEATURE_NAMES),
                    "targets": list(TARGET_COLS),
                },
            },
            CKPT_OUT_PATH,
        )

    print(
        f"  Epoka {epoch:02d}/{train_cfg.epochs:02d} ({epoch_sec:.1f}s) | lr={cur_lr:.2e} | "
        f"train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | "
        f"NDVI MAE={val_metrics['ndvi_mae']:.4f} | NDMI MAE={val_metrics['ndmi_mae']:.4f}{best_tag}"
    )

total_train_sec = time.perf_counter() - t_train_start
history_df = pd.DataFrame(history_rows)
history_csv_path = LOCAL_OUT_DIR / "training_history.csv"
history_df.to_csv(history_csv_path, index=False)

summary_payload = {
    "run_timestamp": RUN_TIMESTAMP,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "train_mode": TRAIN_MODE,
    "entity_col": ENTITY_COL,
    "anchor_stride": ANCHOR_STRIDE,
    "n_train_windows": len(train_idx),
    "n_val_windows": len(val_idx),
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "best_val_metrics": best_metrics,
    "total_train_sec": round(total_train_sec, 2),
    "checkpoint_file": CKPT_FILENAME,
    "model_config": model_cfg.to_dict(),
    "train_config": train_cfg.to_dict(),
}
summary_json_path = LOCAL_OUT_DIR / "training_summary.json"
summary_json_path.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(
    f"\n[GOTOWE] Trening zakończony w {total_train_sec:.1f}s | "
    f"Najlepsza epoka={best_epoch}, val_loss={best_val_loss:.5f} -> {CKPT_OUT_PATH}"
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss", marker="o", markersize=3)
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss", marker="o", markersize=3)
axes[0].axvline(best_epoch, color="green", linestyle="--", alpha=0.7, label=f"best (ep. {best_epoch})")
axes[0].set_title(f"Strata Smooth L1 ({TRAIN_MODE})")
axes[0].set_xlabel("Epoka")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["ndvi_mae"], label="Val NDVI MAE", marker="o", markersize=3)
axes[1].plot(history_df["epoch"], history_df["ndmi_mae"], label="Val NDMI MAE", marker="s", markersize=3)
axes[1].axvline(best_epoch, color="green", linestyle="--", alpha=0.7, label=f"best (ep. {best_epoch})")
axes[1].set_title("Błąd walidacyjny MAE")
axes[1].set_xlabel("Epoka")
axes[1].set_ylabel("MAE")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
curves_png_path = LOCAL_OUT_DIR / "learning_curves.png"
fig.savefig(curves_png_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Zapisano wykres krzywych uczenia -> {curves_png_path}")

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

from ml.config import ModelConfig
from ml.dataset import FeatureScaler
from ml.model import VegetationLSTM

ckpt = torch.load(CKPT_OUT_PATH, map_location=DEVICE, weights_only=True)
loaded_cfg = ModelConfig.from_dict(ckpt["model_config"])
loaded_scaler = FeatureScaler.from_dict(ckpt["scaler"])
loaded_model = VegetationLSTM(loaded_cfg).to(DEVICE)
loaded_model.load_state_dict(ckpt["model_state_dict"])
loaded_model.eval()

dataset.scaler = loaded_scaler
step = max(1, len(val_idx) // 500)
eval_val_idx = val_idx[::step][:500]
eval_loader = DataLoader(Subset(dataset, eval_val_idx), batch_size=256, shuffle=False)

eval_preds, eval_targets, eval_masks = [], [], []
with torch.inference_mode():
    for batch in eval_loader:
        pred = loaded_model(
            batch["x_hist"].to(DEVICE),
            batch["x_future"].to(DEVICE),
            batch["x_static"].to(DEVICE),
            batch["last_targets"].to(DEVICE),
        )
        eval_preds.append(pred.cpu())
        eval_targets.append(batch["y_target"])
        eval_masks.append(batch["y_mask"])

preds_t = torch.cat(eval_preds, dim=0)     # (N, F, 2)
targets_t = torch.cat(eval_targets, dim=0) # (N, F, 2)
masks_t = torch.cat(eval_masks, dim=0)     # (N, F, 2)

horizon_rows = []
for d_idx in range(loaded_cfg.forecast_days):
    m_day = masks_t[:, d_idx, 0] > 0
    cnt = int(m_day.sum().item())
    if cnt > 0:
        ndvi_mae_d = float((preds_t[m_day, d_idx, 0] - targets_t[m_day, d_idx, 0]).abs().mean().item())
        ndmi_mae_d = float((preds_t[m_day, d_idx, 1] - targets_t[m_day, d_idx, 1]).abs().mean().item())
    else:
        ndvi_mae_d = float("nan")
        ndmi_mae_d = float("nan")
    horizon_rows.append(
        {
            "horizon_day": d_idx + 1,
            "valid_obs_count": cnt,
            "ndvi_mae": round(ndvi_mae_d, 4) if np.isfinite(ndvi_mae_d) else np.nan,
            "ndmi_mae": round(ndmi_mae_d, 4) if np.isfinite(ndmi_mae_d) else np.nan,
        }
    )

horizon_df = pd.DataFrame(horizon_rows)
horizon_csv_path = LOCAL_OUT_DIR / "val_horizon_metrics.csv"
horizon_df.to_csv(horizon_csv_path, index=False)
print(f"Weryfikacja checkpointu OK (entity_mode={ckpt.get('entity_mode')}, próbka walidacyjna={len(eval_val_idx)}):")
print(horizon_df.to_string(index=False))

sample_rows = []
n_preview = min(20, len(eval_val_idx))
for k in range(n_preview):
    s_meta = dataset.samples[eval_val_idx[k]]
    for d_idx in range(loaded_cfg.forecast_days):
        sample_rows.append(
            {
                "field_id": s_meta["field_id"],
                "entity_id": s_meta.get("entity_id", s_meta["field_id"]),
                "anchor_date": s_meta["anchor_date"],
                "day_offset": d_idx + 1,
                "ndvi_pred": round(float(preds_t[k, d_idx, 0].item()), 4),
                "ndvi_true": round(float(targets_t[k, d_idx, 0].item()), 4),
                "ndmi_pred": round(float(preds_t[k, d_idx, 1].item()), 4),
                "ndmi_true": round(float(targets_t[k, d_idx, 1].item()), 4),
                "observed_mask": int(masks_t[k, d_idx, 0].item() > 0),
            }
        )

sample_df = pd.DataFrame(sample_rows)
sample_csv_path = LOCAL_OUT_DIR / "sample_val_predictions.csv"
sample_df.to_csv(sample_csv_path, index=False)
print(f"Zapisano próbkę predykcji ({len(sample_df)} wierszy) -> {sample_csv_path}")

In [ ]:
from pathlib import Path
import shutil
import zipfile

result_patterns = ("*.pt", "*.csv", "*.json", "*.png")
files_to_export = sorted(
    p for pat in result_patterns for p in LOCAL_OUT_DIR.glob(pat) if p.is_file()
)
if not files_to_export:
    raise RuntimeError(f"Brak plików wynikowych do skopiowania w {LOCAL_OUT_DIR}")

def _fmt_size(path: Path) -> str:
    size_kb = path.stat().st_size / 1024.0
    return f"{size_kb / 1024.0:.2f} MB" if size_kb >= 1024.0 else f"{size_kb:.1f} KB"

LOCAL_RESULTS_ZIP = LOCAL_WORK_DIR / f"wyniki_{TRAIN_MODE}_{RUN_TIMESTAMP}.zip"
with zipfile.ZipFile(LOCAL_RESULTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for src_file in files_to_export:
        zf.write(src_file, arcname=src_file.name)
assert LOCAL_RESULTS_ZIP.exists(), f"Nie udało się utworzyć paczki ZIP: {LOCAL_RESULTS_ZIP}"
print(f"Spakowano lokalnie wszystkie wyniki -> {LOCAL_RESULTS_ZIP} ({_fmt_size(LOCAL_RESULTS_ZIP)})")

if USE_GOOGLE_DRIVE and DRIVE_PROJECT_DIR.exists():
    DRIVE_RUN_DIR = DRIVE_RESULTS_ROOT / f"run_{TRAIN_MODE}_{RUN_TIMESTAMP}"
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

    copied_paths = []
    for src_file in files_to_export:
        dst_file = DRIVE_RUN_DIR / src_file.name
        shutil.copy2(src_file, dst_file)
        assert dst_file.exists(), f"Nie udało się zapisać pliku na Drive: {dst_file}"
        copied_paths.append(dst_file)

    if COPY_TO_CHECKPOINTS:
        drive_ckpt_dir = DRIVE_PROJECT_DIR / "ml" / "checkpoints"
        drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
        drive_ckpt_path = drive_ckpt_dir / CKPT_FILENAME
        shutil.copy2(CKPT_OUT_PATH, drive_ckpt_path)
        assert drive_ckpt_path.exists(), f"Nie udało się zapisać głównego checkpointu: {drive_ckpt_path}"
        copied_paths.append(drive_ckpt_path)

    print(f"[SUKCES] Zrzut wyników na Google Drive zakończony! Folder uruchomienia: {DRIVE_RUN_DIR}")
    for p in copied_paths:
        print(f"  [OK] {p} ({_fmt_size(p)})")
else:
    print(f"[GOTOWE] Tryb bez Google Drive — paczka wyników czeka w: {LOCAL_RESULTS_ZIP}")
    print(f"Najlepszy model (.pt): {CKPT_OUT_PATH} ({_fmt_size(CKPT_OUT_PATH)})")
    if IN_COLAB:
        print("W panelu Pliki po lewej kliknij prawym przyciskiem na plik .zip (lub .pt) i wybierz Pobierz,")
        print("albo odkomentuj ponizsze pobieranie z przegladarki.")

# Opcjonalne pobranie paczki wyników lub modelu bezpośrednio z przeglądarki:
# from google.colab import files
# files.download(str(LOCAL_RESULTS_ZIP))